# Ghost systems and free superfermions

`ghost_systems.py` constructs the free field vertex (super)algebras that appear as ghosts, for example in string theory [FMS1986] and in quantum Hamiltonian reduction [KRW2003]:

* `BosonicGhosts` (also reached as `vertex_algebras.BosonicGhosts`): even fields $\beta_i, \gamma_i$ with OPEs $\beta_i(z)\gamma_j(w) \sim \delta_{ij}/(z-w)$;
* `FermionicGhosts` (also reached as `vertex_algebras.FermionicGhosts`): odd fields $b_i, c_i$ with OPEs $b_i(z)c_j(w) \sim \delta_{ij}/(z-w)$;
* `FreeSuperFermions`: the fields $\Phi_i$ of a basis of a super vector space with a non-degenerate even form, skew-symmetric on the even part and symmetric on the odd part, satisfying $\Phi_i(z)\Phi_j(w) \sim M_{ij}/(z-w)$. These are symplectic bosons and free fermions, Kac–Roan–Wakimoto's neutral free superfermions [KRW2003]. The form is given by its Gram matrix (as for lattice vertex algebras), or by a basis and the form as a function.

Each constructor returns a `UniversalVertexAlgebra` (see `01_getting_started.ipynb`), so elements are computed with exactly as there (`a.ope(b)`, `a.no(b)`, ...), and equal data give the same algebra.

**Relation to Sage.** Sage has Lie conformal algebras called `BosonicGhosts`, `FermionicGhosts`, `Weyl` and `FreeFermions`. The algebras of this notebook are their universal enveloping vertex algebras, with the central element $K$ set to 1: the generators have the same names, and the OPEs here are the λ-brackets there, $[a_\lambda b] = \sum_{n \ge 0} \frac{\lambda^n}{n!}\, a_{(n)} b$ [Kac1998]. Section 5 checks this.

**Conformal vectors.** None is chosen by default. The common ones are given by name with `conformal_vector=`:

| `conformal_vector=` | weights | βγ: $c$ | bc: $c$ |
|---|---|---|---|
| `'symmetric'` | every field $1/2$ | $-1$ | $1$ |
| `'b1c0'` | $\beta, b$: 1; $\gamma, c$: 0 | $2$ | $-2$ |
| `'b0c1'` | $\beta, b$: 0; $\gamma, c$: 1 | $2$ | $-2$ |
| `'degrees'` + `degrees=` | as given (the two fields of a pair add up to 1) | | |

Also `beta_weight=λ` / `b_weight=λ` (any $\lambda$, or a list with one per pair) gives $c = \pm 2(6\lambda^2 - 6\lambda + 1)$ per pair [FMS1986]. Any other conformal vector is given as an element: build the algebra without conformal data, write $T$ in its generators and pass it as `conformal_vector=T` (section 7). `degrees=` alone gives weights without a conformal vector, and `horizontal_grading='charge'` adds the charge grading ($\beta, b$: $+1$; $\gamma, c$: $-1$).

**Prerequisites:** `01_getting_started.ipynb`.

**Contents**

1. The βγ system and its conformal vectors
2. Weight spaces, characters and the charge grading
3. Several pairs
4. The bc system
5. The same algebras as Sage's Lie conformal algebras
6. Free superfermions from a Gram matrix
7. Prescribed weights and general conformal vectors
8. From a basis and a form: the neutral fermions of a reduction
9. Tensor products of ghost systems
10. Unique representation and pickling
11. Affine $\mathfrak{sl}_2$ at level $-1/2$ inside the βγ system

In [1]:
from vertex_algebras import *
from vertex_algebras.ghost_systems import BosonicGhosts, FermionicGhosts, FreeSuperFermions
import vertex_algebras as va

## 1. The βγ system and its conformal vectors

`BosonicGhosts(R)` is the βγ system of one pair over the ring `R`: two even generators, named `beta` and `gamma`, with the OPE $\beta(z)\gamma(w) \sim 1/(z-w)$. The other orientation follows by skew-symmetry, $\gamma(z)\beta(w) \sim -1/(z-w)$, and $\beta$, $\gamma$ have regular OPEs with themselves.

With no further arguments, nothing beyond the OPEs is chosen: the algebra has no conformal vector, no conformal weights and no grading. The cell below prints the algebra, the three OPEs of the generators, their parities and the (absent) conformal vector.

In [2]:
V = BosonicGhosts(QQ)
print(V)
beta, gamma = V.gens()
print('beta gamma:', beta.ope(gamma), '  gamma beta:', gamma.ope(beta), '  beta beta:', beta.ope(beta))
print('parities:', V.parities(), '  Virasoro vector:', V.conformal_vector())

Universal vertex algebra on generators ['beta', 'gamma'] over Rational Field
beta gamma: [1]   gamma beta: [-1]   beta beta: []
parities: {'beta': 0, 'gamma': 0}   Virasoro vector: None


The βγ system has a one-parameter family of conformal vectors,

$$T_\lambda = (\lambda - 1)\,{:}\partial\beta\,\gamma{:} + \lambda\,{:}\beta\,\partial\gamma{:},$$

for which $\beta$ has conformal weight $\lambda$ and $\gamma$ has weight $1 - \lambda$ [FMS1986]. Three members of the family have names: `'symmetric'` ($\lambda = 1/2$), `'b1c0'` ($\lambda = 1$) and `'b0c1'` ($\lambda = 0$). For each, the cell below prints the weights, the central charge and $T$ itself in the basis of normally ordered monomials.

In [3]:
for name in ('symmetric', 'b1c0', 'b0c1'):
    W = BosonicGhosts(QQ, conformal_vector=name)
    print('%-10s weights %-28s c = %s   T = %s' % (name, W.degrees(), W.central_charge(), W.conformal_vector()))

symmetric  weights {'beta': 1/2, 'gamma': 1/2}  c = -1   T = -1/2*:dbeta gamma: + 1/2*:dgamma beta:
b1c0       weights {'beta': 1, 'gamma': 0}      c = 2   T = :dgamma beta:
b0c1       weights {'beta': 0, 'gamma': 1}      c = 2   T = -:dbeta gamma:


The other members of the family are reached with `beta_weight=λ`, for any $\lambda$ in the coefficient ring. The central charge is $c = 2(6\lambda^2 - 6\lambda + 1)$; for $\lambda = 3/2$ this is the superghost system of the superstring, with $c = 11$ [FMS1986].

The weight can also be a parameter: over the fraction field $\mathbb Q(\lambda)$ (the variable `l` below) the central charge and the weights are computed as functions of $\lambda$.

In [4]:
print([(l, BosonicGhosts(QQ, beta_weight=l).central_charge()) for l in (0, 1/2, 1, 3/2, 2, -1/3)])
K.<l> = Frac(QQ['l'])
Vl = BosonicGhosts(K, beta_weight=l)
print(Vl.central_charge(), Vl.degrees())

[(0, 2), (1/2, -1), (1, 2), (3/2, 11), (2, 26), (-1/3, 22/3)]
12*l^2 - 12*l + 2 {'beta': l, 'gamma': -l + 1}


## 2. Weight spaces, characters and the charge grading

Once the generators have conformal weights, `V.weight_space(n)` and `V.character(N)` are available (see `01_getting_started.ipynb`, section 5). How they behave depends on $\lambda$.

With the symmetric conformal vector both fields have weight $1/2$, so $\partial^n\beta$ and $\partial^n\gamma$ have weight $n + 1/2 > 0$. Every weight space is then finite-dimensional, and the character is

$$\prod_{n\ge1} \frac{1}{(1-q^{n-1/2})^{2}}.$$

The cell below prints the character and compares the dimensions of the weight spaces with the coefficients of this product, written in the variable $t = q^{1/2}$.

In [5]:
V12 = BosonicGhosts(QQ, conformal_vector='symmetric')
print(V12.character(4))
P.<t> = PowerSeriesRing(QQ, default_prec=8)          # t = q^(1/2)
print([V12.weight_space(k / 2).dimension() for k in range(8)], list(prod((1 - t^(2*n - 1))^-2 for n in range(1, 5)))[:8])

1 + 2*q^(1/2) + 3*q + 6*q^(3/2) + 9*q^2 + 14*q^(5/2) + 22*q^3 + 32*q^(7/2) + O(q^4)
[1, 2, 3, 6, 9, 14, 22, 32] [1, 2, 3, 6, 9, 14, 22, 32]


With `'b1c0'`, $\gamma$ has weight 0. Multiplying by $\gamma$ does not change the weight, so every nonzero weight space contains infinitely many monomials (for example $\beta, {:}\beta\gamma{:}, {:}\beta\gamma\gamma{:}, \ldots$ in weight 1) and the character does not exist.

The remedy is a second grading. `horizontal_grading='charge'` grades the algebra by charge: $\beta$ and $b$ have charge $+1$, $\gamma$ and $c$ have charge $-1$, and the charge is additive on normally ordered products. The subspace of a given weight *and* charge is finite-dimensional, and is obtained with `V.weight_space(n, horizontal_grade=g)`. The cell below checks that the weight-1 space is infinite-dimensional, prints the charges of the generators, and then the dimensions of the weight-1 spaces of charges $-2, \ldots, 2$.

In [6]:
V1 = BosonicGhosts(QQ, conformal_vector='b1c0')
print(V1.weight_space(1).is_finite_dimensional())
V1c = BosonicGhosts(QQ, conformal_vector='b1c0', horizontal_grading='charge')
print(V1c.gen('gamma').horizontal_grade(), V1c.gen('beta').horizontal_grade())
print([V1c.weight_space(1, horizontal_grade=g).dimension() for g in (-2, -1, 0, 1, 2)])
print(V1c.weight_space(1, horizontal_grade=0).basis())

False
-1 1
[2, 2, 1, 1, 0]
[:beta gamma:]


A monomial of charge 0 contains as many $\beta$'s as $\gamma$'s. Each pair $\partial^n\beta$, $\partial^m\gamma$ contributes the weight $(\lambda + n) + (1 - \lambda + m) = 1 + n + m$, in which $\lambda$ cancels. So the dimensions of the charge-0 weight spaces do not depend on the choice of conformal vector, as the cell below confirms for the three named ones.

In [7]:
for name in ('symmetric', 'b1c0', 'b0c1'):
    W = BosonicGhosts(QQ, conformal_vector=name, horizontal_grading='charge')
    print('%-10s' % name, [W.weight_space(n, horizontal_grade=0).dimension() for n in range(7)])

symmetric  [1, 1, 3, 6, 12, 21, 38]
b1c0       [1, 1, 3, 6, 12, 21, 38]
b0c1       [1, 1, 3, 6, 12, 21, 38]


## 3. Several pairs

`rank=n` gives $n$ pairs, with $\beta_i(z)\gamma_j(w) \sim \delta_{ij}/(z-w)$; fields of different pairs have regular OPEs. The default names are Sage's, `beta0, beta1, ..., gamma0, gamma1, ...`. Other names are given with `names=`, which lists the first fields of all pairs and then the second ones: with `names=['u', 'v', 'x', 'y']` the pairs are $(u, x)$ and $(v, y)$.

The weights can differ from pair to pair. There are two equivalent ways to give them: a list to `beta_weight=`, with one $\lambda$ per pair, or `conformal_vector='degrees'` with a dict `degrees=` of all the weights, in which the two weights of each pair add up to 1. The central charges of the pairs add up. The cell below builds the same rank-3 algebra in both ways, and then a rank-2 algebra with its own names.

In [8]:
V3 = BosonicGhosts(QQ, rank=3, beta_weight=[0, 1/2, 1])
print(V3.variable_names(), V3.central_charge())
print(V3.degrees())
Vw = BosonicGhosts(QQ, rank=3, conformal_vector='degrees',
                   degrees={'beta0': 0, 'beta1': 1/2, 'beta2': 1, 'gamma0': 1, 'gamma1': 1/2, 'gamma2': 0})
print(Vw is V3)
Vn = BosonicGhosts(QQ, rank=2, names=['u', 'v', 'x', 'y'], conformal_vector='symmetric')
print(Vn.gen('u').ope(Vn.gen('x')), Vn.gen('u').ope(Vn.gen('y')))

('beta0', 'beta1', 'beta2', 'gamma0', 'gamma1', 'gamma2') 3
{'beta0': 0, 'beta1': 1/2, 'beta2': 1, 'gamma0': 1, 'gamma1': 1/2, 'gamma2': 0}
True
[1] []


## 4. The bc system

`FermionicGhosts(R)` is the fermionic counterpart: two odd generators `b` and `c` with $b(z)c(w) \sim 1/(z-w)$. Since both fields are odd, skew-symmetry gives the same sign in the other orientation, $c(z)b(w) \sim 1/(z-w)$. The arguments are those of `BosonicGhosts`, with `b_weight=λ` in place of `beta_weight=λ`: then $b$ has weight $\lambda$, $c$ has weight $1 - \lambda$, and the central charge has the opposite sign, $c = -2(6\lambda^2 - 6\lambda + 1)$. For $\lambda = 2$ these are the reparametrization ghosts of the bosonic string, with $c = -26$ [FMS1986].

Graded by charge, the bc system also illustrates the boson–fermion correspondence [Kac1998]: its charge-0 part is a Heisenberg Fock space, so the character at charge 0 is $1/(q)_\infty = \prod_{n \ge 1} (1 - q^n)^{-1}$, the generating function of partitions, for every $\lambda$.

The cell below prints the OPEs and parities, the central charges of the named conformal vectors and of $\lambda = 3/2, 2, 3$, and the charge-0 character for $\lambda = 2$. Its last line checks that the catalog entries `va.FermionicGhosts` and `va.BosonicGhosts` are the same functions as those imported from `ghost_systems`.

In [9]:
B = FermionicGhosts(QQ)
b, c = B.gens()
print('b c:', b.ope(c), '  c b:', c.ope(b), '  b b:', b.ope(b), '  parities:', B.parities())
print([(name, FermionicGhosts(QQ, conformal_vector=name).central_charge()) for name in ('symmetric', 'b1c0', 'b0c1')])
print([(l, FermionicGhosts(QQ, b_weight=l).central_charge()) for l in (3/2, 2, 3)])
B2 = FermionicGhosts(QQ, b_weight=2, horizontal_grading='charge')
print(B2.degrees(), B2.character(7, horizontal_grade=0))
print(va.FermionicGhosts is FermionicGhosts, va.BosonicGhosts is BosonicGhosts)

b c: [1]   c b: [1]   b b: []   parities: {'b': 1, 'c': 1}
[('symmetric', 1), ('b1c0', -2), ('b0c1', -2)]
[(3/2, -11), (2, -26), (3, -74)]
{'b': 2, 'c': -1} 1 + q + 2*q^2 + 3*q^3 + 5*q^4 + 7*q^5 + 11*q^6 + O(q^7)
True True


Zhu's algebra [Zhu1996] is computed with `V.zhu_algebra()` (see `09_zhu_algebras.ipynb`). For the $bc$ system with weights 1 and 0 it is generated by the classes $[b]$ and $[c]$, which satisfy $[b][c] + [c][b] = 1$ and $[b]^2 = 0$: the Clifford algebra on $[b], [c]$.

In [10]:
A = FermionicGhosts(QQ, conformal_vector='b1c0').zhu_algebra()
zb, zc = A.gens()
zb * zc + zc * zb, zb * zb

(1, 0)

## 5. The same algebras as Sage's Lie conformal algebras

Sage's catalog `lie_conformal_algebras` contains `BosonicGhosts`, `FermionicGhosts`, `Weyl` and `FreeFermions`. Each is a Lie conformal algebra with a central element $K$, and `universal_enveloping_vertex_algebra(L, central_parameters={'K': 1})` is its universal enveloping vertex algebra with $K$ acting as 1 [Kac1998].

The cell below checks that these are not merely isomorphic to the algebras of this notebook but the *same objects* (`is`), once the names and weights are matched: Sage gives $\beta$ and $b$ weight 1 and $\gamma$, $c$ weight 0, `Weyl` is the symplectic pair of `FreeSuperFermions` (section 6) with Sage's names `alpha0, alpha1`, and `FreeFermions` has fields of weight $1/2$ named `psi_0, psi_1`.

In [11]:
from vertex_algebras.lie_conformal import universal_enveloping_vertex_algebra as U
K1 = {'K': 1}
print(U(lie_conformal_algebras.BosonicGhosts(QQ), central_parameters=K1)
      is BosonicGhosts(QQ, degrees={'beta': 1, 'gamma': 0}))
print(U(lie_conformal_algebras.FermionicGhosts(QQ), central_parameters=K1)
      is FermionicGhosts(QQ, degrees={'b': 1, 'c': 0}))
print(U(lie_conformal_algebras.Weyl(QQ), central_parameters=K1)
      is FreeSuperFermions(QQ, matrix([[0, 1], [-1, 0]]), names=['alpha0', 'alpha1']))
M = matrix([[0, 1], [1, 0]])
print(U(lie_conformal_algebras.FreeFermions(QQ, gram_matrix=M), central_parameters=K1)
      is FreeSuperFermions(QQ, M, names=['psi_0', 'psi_1'], degrees={'psi_0': 1/2, 'psi_1': 1/2}))

True
True
True
True


## 6. Free superfermions from a Gram matrix

`FreeSuperFermions(R, M)` takes the Gram matrix $M$ of the form and builds generators $\Phi_i$ with $\Phi_i(z)\Phi_j(w) \sim M_{ij}/(z-w)$ [KRW2003]. The default names are `Phi0, Phi1, ...`; `names=` takes a prefix or a list.

As for lattice vertex algebras, the parities are not given separately but read off the matrix. Skew-symmetry of the OPE forces $M$ to be skew-symmetric between even fields and symmetric between odd fields. So a block of basis vectors connected by nonzero entries is even if $M$ is skew-symmetric on it (symplectic bosons) and odd if $M$ is symmetric on it (fermions). The basis vectors of the two kinds may come in any order.

With `conformal_vector='symmetric'` every field has weight $1/2$ and the central charge is $c = -\operatorname{sdim}/2$, where the superdimension counts even fields with $+1$ and odd fields with $-1$: each symplectic boson contributes $-1/2$ and each fermion $+1/2$. The cell below shows four examples: a single (Majorana) fermion, a symplectic pair, a fermion pair, and a mixed form with its even and odd basis vectors interleaved.

In [12]:
examples = {'Majorana fermion': matrix([[1]]), 'symplectic pair': matrix([[0, 1], [-1, 0]]),
            'fermion pair': matrix([[0, 1], [1, 0]]),
            'mixed, any order': matrix([[0, 0, 1, 0], [0, 2, 0, 0], [-1, 0, 0, 0], [0, 0, 0, -3]])}
for name, M in examples.items():
    F = FreeSuperFermions(QQ, M, conformal_vector='symmetric')
    print('%-18s parities %-44s c = %s' % (name, F.parities(), F.central_charge()))

Majorana fermion   parities {'Phi0': 1}                                  c = 1/2
symplectic pair    parities {'Phi0': 0, 'Phi1': 0}                       c = -1
fermion pair       parities {'Phi0': 1, 'Phi1': 1}                       c = 1
mixed, any order   parities {'Phi0': 0, 'Phi1': 1, 'Phi2': 0, 'Phi3': 1} c = 0


The ghost systems are special cases. A symplectic pair with the names `beta`, `gamma` and the symmetric conformal vector is the βγ system with weights $1/2$, and a fermion pair named `b`, `c` is the bc system with weights $1/2$: the cell below checks that they are the same objects. Its last lines show a form that is not normalized, $\Phi_0(z)\Phi_1(w) \sim 2/(z-w)$, with a name prefix; the symmetric conformal vector takes the normalization into account.

In [13]:
J, S = matrix([[0, 1], [-1, 0]]), matrix([[0, 1], [1, 0]])
print(FreeSuperFermions(QQ, J, names=['beta', 'gamma'], conformal_vector='symmetric')
      is BosonicGhosts(QQ, conformal_vector='symmetric'))
print(FreeSuperFermions(QQ, S, names=['b', 'c'], conformal_vector='symmetric')
      is FermionicGhosts(QQ, conformal_vector='symmetric'))
F = FreeSuperFermions(QQ, matrix([[0, 2], [-2, 0]]), names='x', conformal_vector='symmetric')
print(F.variable_names(), F.gen('x0').ope(F.gen('x1')), F.conformal_vector())

True
True
('x0', 'x1') [2] -1/4*:dx0 x1: + 1/4*:dx1 x0:


Forms that do not define such an algebra are refused with a `ValueError`. The three matrices below are, in order: one that is neither skew-symmetric nor symmetric on a block; one that mixes both on a connected block ($\Phi_0, \Phi_1$ are paired skew-symmetrically, but $\Phi_0, \Phi_2$ symmetrically, so $\Phi_0$ would have to be both even and odd); and a degenerate one.

In [14]:
for M in (matrix([[0, 1], [2, 0]]), matrix([[0, 1, 1], [-1, 0, 0], [1, 0, 1]]), matrix([[1, 1], [1, 1]])):
    try:
        FreeSuperFermions(QQ, M)
    except ValueError as err:
        print('ValueError:', err)

ValueError: the form must be, on each block of basis vectors it connects, skew-symmetric (even) or symmetric (odd)
ValueError: the form must be, on each block of basis vectors it connects, skew-symmetric (even) or symmetric (odd)
ValueError: the form must be non-degenerate


## 7. Prescribed weights and general conformal vectors

`conformal_vector='degrees'` with a dict `degrees=` builds the conformal vector for which the generators have exactly the given weights. It has the form $T = T_{\mathrm{sym}} + \partial J$, where $T_{\mathrm{sym}}$ is the symmetric conformal vector and $J$ is quadratic in the fields. Not all weights are possible: the weights of paired fields ($M_{ij} \ne 0$) must add up to 1, because their OPE has a simple pole with a scalar coefficient. In particular a Majorana fermion, which is paired with itself, has weight $1/2$.

The example below combines a βγ pair with $\lambda = 2$ ($c = 26$), a bc pair with $\lambda = 3/2$ ($c = -11$) and a Majorana fermion ($c = 1/2$), so $c = 26 - 11 + 1/2 = 31/2$. Asking for weight 1 for the Majorana fermion instead is refused.

In [15]:
Mb = block_diagonal_matrix([J, S, matrix([[1]])])
w = {'Phi0': 2, 'Phi1': -1, 'Phi2': 3/2, 'Phi3': -1/2, 'Phi4': 1/2}
Wf = FreeSuperFermions(QQ, Mb, conformal_vector='degrees', degrees=w)
print(Wf.degrees(), Wf.central_charge())
try:
    FreeSuperFermions(QQ, Mb, conformal_vector='degrees', degrees=dict(w, Phi4=1))
except ValueError as err:
    print('ValueError:', err)

{'Phi0': 2, 'Phi1': -1, 'Phi2': 3/2, 'Phi3': -1/2, 'Phi4': 1/2} 31/2
ValueError: the weights of paired fields (M_ij != 0) must add up to 1


A conformal vector that is not in these families is given as an element, as for lattice vertex algebras. First build the algebra without conformal data, write $T$ in its generators, and then pass it as `conformal_vector=T`; the constructor checks that $T$ is a conformal vector (see `01_getting_started.ipynb`, section 4).

Below, $T = \frac12\,{:}\partial\beta\,\gamma{:} + \frac32\,{:}\beta\,\partial\gamma{:}$ is written by hand on a symplectic pair. This is $T_\lambda$ of section 1 for $\lambda = 3/2$, and the result is the same object as `BosonicGhosts(QQ, beta_weight=3/2)`.

In [16]:
P0 = FreeSuperFermions(QQ, J, names=['beta', 'gamma'])
pb, pg = P0.gens()
T = pb.derivative().no(pg) / 2 + 3/2 * pb.no(pg.derivative())
G = FreeSuperFermions(QQ, J, names=['beta', 'gamma'], conformal_vector=T)
print(G.degrees(), G.central_charge(), G is BosonicGhosts(QQ, beta_weight=3/2))

{'beta': 3/2, 'gamma': -1/2} 11 True


## 8. From a basis and a form: the neutral fermions of a reduction

Instead of a Gram matrix, `FreeSuperFermions(R, basis, form)` takes a list of elements of a super module and the form as a function of two elements. The parities are those of the elements, checked against the symmetry of the form, and the generators are named after the elements as they print.

This is the form in which free superfermions appear in the quantum Hamiltonian reduction of Kac–Roan–Wakimoto [KRW2003]. For a Lie superalgebra $\mathfrak g$ with a $\frac12\mathbb Z$-grading and a nilpotent $f \in \mathfrak g_{-1}$, the reduction uses neutral free superfermions on $\mathfrak g_{1/2}$, with the form $\langle a, b\rangle = (f\,|\,[a, b])$.

The function `neutral_fermions` below does this for the minimal grading, by $x = \theta/2$ with $\theta$ the root `g.theta()` (see `02_lie_superalgebras.ipynb` for `LieSuperalgebra`): $f = e_{-\theta}$, and $\mathfrak g_{1/2}$ is spanned by the root vectors of the roots $\beta$ with $(\beta|\theta) = 1$. For $\mathfrak{sl}(2|1)$ this gives two fermions; the reduction is the $N = 2$ superconformal algebra [KRW2003, KW2004].

In [17]:
from vertex_algebras.lie_superalgebras import LieSuperalgebra


def neutral_fermions(g):
    th = g.theta()
    f = g.root_vector(th, -1)
    half = [g.basis()[k] for k in g.basis().keys()
            if any(g.root_on_basis(k)) and g.root_norm(g.root_on_basis(k), th) == 1]
    return half, FreeSuperFermions(QQ, half, lambda a, b: g.invariant_form(f, a.bracket(b)),
                                   conformal_vector='symmetric')


g = LieSuperalgebra(QQ, 'sl(2|1)')
half, Ne = neutral_fermions(g)
print(half)
print(Ne.parities(), Ne.central_charge())
print([x.ope(y) for x in Ne.gens() for y in Ne.gens()])

[E[alpha[1] + alpha[2]], E[-alpha[2]]]
{'E[alpha[1] + alpha[2]]': 1, 'E[-alpha[2]]': 1} 1
[[], [-1], [-1], []]


The same function applied to further minimal reductions. For $\mathfrak{sl}_3$ there are two symplectic bosons, the ones of the Bershadsky–Polyakov algebra [Ber1991, Pol1990] (constructed in `08_w_algebras.ipynb`); for $\mathfrak{osp}(1|2)$ there is one fermion, that of the Neveu–Schwarz algebra. The numbers of even and odd elements of $\mathfrak g_{1/2}$ agree with the tables of Kac–Wakimoto [KW2004].

In every case the form $(f|[\cdot,\cdot])$ is non-degenerate (otherwise the constructor would refuse it), and the central charge $c = -\operatorname{sdim}\mathfrak g_{1/2}/2$ of the symmetric conformal vector is the contribution of the neutral fermions to the central charge of the W-algebra [KRW2003].

In [18]:
for label, spec, kw in [('sl_3 (Bershadsky-Polyakov)', CartanMatrix(['A', 2]), {}),
                        ('osp(1|2) = spo(2|1)', 'spo(2|1)', {}), ('D(2,1;2)', 'D(2,1;a)', {'a': 2}),
                        ('F(4), θ in sl_2', 'F(4)', {'theta': 'sl2'}), ('G(3), θ in sl_2', 'G(3)', {'theta': 'sl2'}),
                        ('osp(5|2)', 'osp(5|2)', {})]:
    g = LieSuperalgebra(QQ, spec, **kw)
    half, Ne = neutral_fermions(g)
    even = sum(1 for x in half if not x.is_even_odd())
    print('%-28s g_1/2: %d even, %d odd   c = %s' % (label, even, len(half) - even, Ne.central_charge()))

sl_3 (Bershadsky-Polyakov)   g_1/2: 2 even, 0 odd   c = -1
osp(1|2) = spo(2|1)          g_1/2: 0 even, 1 odd   c = 1/2


D(2,1;2)                     g_1/2: 0 even, 4 odd   c = 2
F(4), θ in sl_2              g_1/2: 0 even, 8 odd   c = 4


G(3), θ in sl_2              g_1/2: 0 even, 7 odd   c = 7/2

osp(5|2)                     g_1/2: 2 even, 4 odd   c = 1


## 9. Tensor products of ghost systems

Ghost systems are ordinary vertex algebras of this library, so they combine with other vertex algebras: `A.tensor(B, conformal_vector='factors')` is the tensor product with the sum of the conformal vectors of the factors, and the central charges add.

The ghosts of the superstring [FMS1986] are the bc system with $\lambda = 2$ ($c = -26$) together with the βγ system with $\lambda = 3/2$ ($c = 11$), in total $c = -15$.

In [19]:
ghosts = BosonicGhosts(QQ, beta_weight=3/2).tensor(FermionicGhosts(QQ, b_weight=2), conformal_vector='factors')
print(ghosts.central_charge(), ghosts.degrees())

-15 {'beta': 3/2, 'gamma': -1/2, 'b': 2, 'c': -1}


## 10. Unique representation and pickling

The constructors have unique representation: two calls that describe the same data return the same object, even if they describe it differently. For instance `conformal_vector='b1c0'` and `beta_weight=1` give identical algebras. This is what made the checks with `is` in the earlier sections possible. The algebras can be saved and loaded (`dumps`, `loads`), which also returns the same object, and they pass Sage's generic tests (`TestSuite`).

In [20]:
print(BosonicGhosts(QQ, conformal_vector='b1c0') is BosonicGhosts(QQ, beta_weight=1))
print(loads(dumps(Wf)) is Wf, loads(dumps(Ne)) is Ne)
TestSuite(Wf).run(); TestSuite(B2).run()
print('TestSuite ok')

True


True True


TestSuite ok


## 11. Affine $\mathfrak{sl}_2$ at level $-1/2$ inside the βγ system

Free fields are used to realise other vertex algebras. The basic example for the βγ system [FF1985]: the three quadratic fields

$$e = \tfrac12\,{:}\gamma\gamma{:}, \qquad h = {:}\beta\gamma{:}, \qquad f = -\tfrac12\,{:}\beta\beta{:}$$

satisfy the OPEs of affine $\mathfrak{sl}_2$ at level $k = -1/2$,

$$h(z)e(w) \sim \frac{2e(w)}{z-w}, \quad h(z)f(w) \sim \frac{-2f(w)}{z-w}, \quad h(z)h(w) \sim \frac{2k}{(z-w)^2}, \quad e(z)f(w) \sim \frac{k}{(z-w)^2} + \frac{h(w)}{z-w}.$$

In other words, sending the generators $e, h, f$ of the universal affine vertex algebra $V^{-1/2}(\mathfrak{sl}_2)$ (see `03_affine_vertex_algebras.ipynb`; its generators are named `alpha[1]`, `alphacheck[1]`, `-alpha[1]`) to these fields defines a morphism of vertex algebras. `V.hom(images, W)` builds the morphism and checks that the images satisfy all OPEs of the generators of `V`; since `V` is a universal vertex algebra this check is exact, and `checked_up_to_weight()` is `None`.

In [21]:
Bg = BosonicGhosts(QQ, conformal_vector='symmetric')
beta, gamma = Bg.gens()
Vk = va.Affine(QQ, ['A', 1], -1/2, conformal_vector='sugawara')
e, h, f = Vk.gens()
phi = Vk.hom([gamma.no(gamma)/2, beta.no(gamma), -beta.no(beta)/2], Bg)
print(phi)
print(phi.checked_up_to_weight())

Vertex algebra morphism:
  From: Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field
  To:   Universal vertex algebra on generators ['beta', 'gamma'] over Rational Field
  Defn: alpha[1] |--> 1/2*:gamma gamma:
        alphacheck[1] |--> :beta gamma:
        -alpha[1] |--> -1/2*:beta beta:
None


Images that do not satisfy the OPEs are refused with a `ValueError` that names the first OPE that fails. Below, first the wrong sign for $f$, and then the same fields as images of the generators of $V^{1}(\mathfrak{sl}_2)$: the level is fixed by the double contractions in $h(z)h(w)$ and $e(z)f(w)$, so only $k = -1/2$ is possible.

In [22]:
for level, images in [(-1/2, [gamma.no(gamma)/2, beta.no(gamma), beta.no(beta)/2]),
                      (1, [gamma.no(gamma)/2, beta.no(gamma), -beta.no(beta)/2])]:
    try:
        va.Affine(QQ, ['A', 1], level).hom(images, Bg)
    except ValueError as err:
        print('ValueError:', err)

ValueError: the images do not satisfy the OPE of alpha[1] and -alpha[1]: pole 0 is -:beta gamma: instead of :beta gamma:
ValueError: the images do not satisfy the OPE of alpha[1] and -alpha[1]: pole 1 is -1/2 instead of 1


The morphism respects the conformal structures. The Sugawara vector at level $-1/2$ has central charge $3k/(k+2) = -1$, the central charge of the symmetric conformal vector of the βγ system, and $\varphi$ maps the one to the other: the images $e, h, f$ have weight 1 exactly when $\beta$ and $\gamma$ have weight $1/2$, which is why this section uses $\lambda = 1/2$.

In [23]:
print(Vk.central_charge(), Bg.central_charge())
print(phi(Vk.conformal_vector()) == Bg.conformal_vector(), phi.is_conformal())

-1 -1
True True


The morphism is neither injective nor surjective. Its kernel starts in weight 4, with a 5-dimensional space: the $\mathfrak{sl}_2$-module generated by the singular vector of weight 4 of $V^{-1/2}(\mathfrak{sl}_2)$, which generates the maximal ideal ($-1/2$ is an admissible level [KW1988]). So $\varphi$ factors through the simple quotient $L_{-1/2}(\mathfrak{sl}_2)$. Its image consists of the monomials with an even number of fields, which are those of integer weight: the last line compares the dimensions of the image with those of the integer weight spaces of the βγ system. This identifies $L_{-1/2}(\mathfrak{sl}_2)$ with the even part of the βγ system; see [Rid2009] for its representation theory.

In [24]:
Ker = phi.kernel(); Im = phi.image()
print([Ker.weight_space(w).dimension() for w in range(6)])
print([Im.weight_space(w).dimension() for w in range(6)], [Bg.weight_space(w).dimension() for w in range(6)])

[0, 0, 0, 0, 5, 15]
[1, 3, 9, 22, 46, 93] [1, 3, 9, 22, 46, 93]


## References

- [FMS1986] D. Friedan, E. Martinec, S. Shenker, *Conformal invariance, supersymmetry and string theory*, Nuclear Phys. B 271 (1986), 93–165, [doi:10.1016/0550-3213(86)90356-1](https://doi.org/10.1016/0550-3213(86)90356-1). — βγ and bc systems of weight λ, the ghosts of the string and the superstring (sections 1, 4, 9).
- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015). — neutral free superfermions in the reduction, their central charge (sections 6, 8).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011). — the tables of minimal reductions (section 8).
- [Ber1991] M. Bershadsky, *Conformal field theories via Hamiltonian reduction*, Comm. Math. Phys. 139 (1991), 71–82, [doi:10.1007/BF02102729](https://doi.org/10.1007/BF02102729). — the Bershadsky–Polyakov algebra (section 8).
- [Pol1990] A. M. Polyakov, *Gauge transformations and diffeomorphisms*, Internat. J. Modern Phys. A 5 (1990), 833–842, [doi:10.1142/S0217751X90000386](https://doi.org/10.1142/S0217751X90000386). — the Bershadsky–Polyakov algebra (section 8).
- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010). — λ-brackets and universal enveloping vertex algebras of Lie conformal algebras (section 5), the boson–fermion correspondence (section 4).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — Zhu's algebra (section 4).
- [FF1985] A. J. Feingold, I. B. Frenkel, *Classical affine algebras*, Adv. Math. 56 (1985), 117–172, [doi:10.1016/0001-8708(85)90027-1](https://doi.org/10.1016/0001-8708(85)90027-1). — affine algebras realised by quadratics in free fields (section 11).
- [KW1988] V. G. Kac, M. Wakimoto, *Modular invariant representations of infinite-dimensional Lie algebras and superalgebras*, Proc. Nat. Acad. Sci. U.S.A. 85 (1988), 4956–4960, [doi:10.1073/pnas.85.14.4956](https://doi.org/10.1073/pnas.85.14.4956). — admissible levels (section 11).
- [Rid2009] D. Ridout, *$\widehat{\mathfrak{sl}}(2)_{-1/2}$: a case study*, Nuclear Phys. B 814 (2009), 485–521, [arXiv:0810.3532](https://arxiv.org/abs/0810.3532). — affine $\mathfrak{sl}_2$ at level $-1/2$ and the βγ system (section 11).